---
## 1. Configuration et Imports

In [1]:
# Imports nécessaires
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import warnings
warnings.filterwarnings('ignore')

# Chemins
BASE_PATH = '/home/henintsoa/CFIM'
CSV_PATH = os.path.join(BASE_PATH, 'csv')
FINAL_PATH = os.path.join(BASE_PATH, 'final/data')
SHAPES_PATH = os.path.join(BASE_PATH, 'Mdg_COD_Boundry_Shapes_BNGRC_OCHA_Dec17')

print("✅ Configuration chargée")
print(f"📂 Chemin des données: {CSV_PATH}")
print(f"📂 Chemin des shapefiles: {SHAPES_PATH}")

✅ Configuration chargée
📂 Chemin des données: /home/henintsoa/CFIM/csv
📂 Chemin des shapefiles: /home/henintsoa/CFIM/Mdg_COD_Boundry_Shapes_BNGRC_OCHA_Dec17


---
## 2. Chargement des Données d'Incidents

In [2]:
# Charger les données d'incidents
df_incidents = pd.read_csv(os.path.join(CSV_PATH, 'incidents_maritimes_complets_2017_2022.csv'))

print("📌 DONNÉES DES INCIDENTS MARITIMES")
print("=" * 60)
print(f"\n📋 Colonnes: {list(df_incidents.columns)}")
print(f"\n📈 Nombre total d'enregistrements: {len(df_incidents)}")

# Filtrer les vrais incidents (exclure "Aucun incident maritime")
df_vrais_incidents = df_incidents[
    ~df_incidents['description'].str.contains('Aucun incident', na=True)
].copy()

print(f"\n⚠️ Nombre de VRAIS incidents: {len(df_vrais_incidents)}")

# Afficher un aperçu
display(df_vrais_incidents[['Date debut', 'District', 'Commune', 'Region', 'Types', 'Longitude', 'Latitude']].head(10))

📌 DONNÉES DES INCIDENTS MARITIMES

📋 Colonnes: ['Date debut', 'Date fin', 'Thématique', 'Objets', 'District', 'Commune', 'Localite', 'Region', 'Types', 'Longitude', 'Latitude', 'Personne concerne', 'Mort', 'Colonne1', 'description', 'Commentaire', 'Date_debut_dt']

📈 Nombre total d'enregistrements: 2215

⚠️ Nombre de VRAIS incidents: 269


,Date debut,District,Commune,Region,Types,Longitude,Latitude
35,05/02/2017,MADAGASCAR,NaN,NaN,NaN,46.325890,-15.766938
55,25/02/2017,Katsepy Mahajanga/ Madagascar,NaN,NaN,NaN,46.289816,-15.737907
65,07/03/2017,OCEAN INDIEN,NaN,NaN,NaN,54.740555,-21.401068
75,17/03/2017,madagascar,NaN,NaN,NaN,43.872715,-18.042964
89,31/03/2017,OCEAN INDIEN,NaN,NaN,NaN,68.761516,-20.827302
106,17/04/2017,MADAGASCAR,NaN,NaN,NaN,49.420532,-18.150623
109,20/04/2017,MADAGASCAR,NaN,NaN,NaN,50.063335,-13.360009
128,09/05/2017,MADAGASCAR,NaN,NaN,NaN,49.869580,-15.966218
189,09/07/2017,OCEAN INDIEN,NaN,NaN,NaN,59.665862,13.417318
191,11/07/2017,MADAGASCAR,NaN,NaN,NaN,49.502451,-17.514099


In [3]:
# Analyser les coordonnées disponibles
print("📊 ANALYSE DES COORDONNÉES")
print("=" * 60)

# Incidents avec coordonnées valides
has_coords = df_vrais_incidents['Longitude'].notna() & df_vrais_incidents['Latitude'].notna()
n_with_coords = has_coords.sum()
n_without_coords = len(df_vrais_incidents) - n_with_coords

print(f"\n✅ Incidents AVEC coordonnées: {n_with_coords} ({100*n_with_coords/len(df_vrais_incidents):.1f}%)")
print(f"❌ Incidents SANS coordonnées: {n_without_coords} ({100*n_without_coords/len(df_vrais_incidents):.1f}%)")

# Analyser les informations de localisation disponibles pour ceux sans coordonnées
df_sans_coords = df_vrais_incidents[~has_coords].copy()

print(f"\n📍 Pour les {n_without_coords} incidents sans coordonnées:")
print(f"   • Avec District: {df_sans_coords['District'].notna().sum()}")
print(f"   • Avec Commune: {df_sans_coords['Commune'].notna().sum()}")
print(f"   • Avec Region: {df_sans_coords['Region'].notna().sum()}")

📊 ANALYSE DES COORDONNÉES

✅ Incidents AVEC coordonnées: 263 (97.8%)
❌ Incidents SANS coordonnées: 6 (2.2%)

📍 Pour les 6 incidents sans coordonnées:
   • Avec District: 6
   • Avec Commune: 0
   • Avec Region: 0


---
## 3. Chargement des Données Géographiques (Shapefiles)

Nous chargeons les shapefiles des divisions administratives de Madagascar pour le géocodage.

In [4]:
# Charger les shapefiles
print("📂 CHARGEMENT DES SHAPEFILES")
print("=" * 60)

# Régions
gdf_regions = gpd.read_file(os.path.join(SHAPES_PATH, 'mdg_polbnda_adm1_Regions_BNGRC_OCHA.shp'))
print(f"\n🗺️ Régions: {len(gdf_regions)} entités")
print(f"   Colonnes: {list(gdf_regions.columns)}")

# Districts
gdf_districts = gpd.read_file(os.path.join(SHAPES_PATH, 'mdg_polbnda_adm2_Distritcts_BNGRC_OCHA.shp'))
print(f"\n🗺️ Districts: {len(gdf_districts)} entités")
print(f"   Colonnes: {list(gdf_districts.columns)}")

# Communes
gdf_communes = gpd.read_file(os.path.join(SHAPES_PATH, 'mdg_polbnda_adm3_Communes_BNGRC_OCHA.shp'))
print(f"\n🗺️ Communes: {len(gdf_communes)} entités")
print(f"   Colonnes: {list(gdf_communes.columns)}")

📂 CHARGEMENT DES SHAPEFILES

🗺️ Régions: 22 entités
   Colonnes: ['REG_PCODE', 'R_CODE', 'REGION_NAM', 'BNGRC_R_CO', 'BNGRC_REG_', 'REG_FKT_SH', 'PROV_CODE', 'OLD_PROVIN', 'Source', 'Notes', 'Shape_Leng', 'Shape_Area', 'geometry']

🗺️ Districts: 119 entités
   Colonnes: ['DIST_PCODE', 'DISTRICT_N', 'BNGRC_D_CO', 'BNGRC_DIST', 'DIS_FKT', 'REG_PCODE', 'REGION_NAM', 'BNGRC_R_CO', 'BNGRC_REG_', 'REG_FKT_SH', 'C_PROV', 'OLD_PROVIN', 'NOTES', 'OLD_DISTRI', 'SOURCE', 'Shape_Leng', 'Shape_Area', 'geometry']

🗺️ Communes: 1577 entités
   Colonnes: ['COM_PCODE', 'C_CODE', 'COMMUNE_NA', 'BNGRC_COM_', 'DIST_PCODE', 'DISTRICT_N', 'BNGRC_D_CO', 'BNGRC_DIS_', 'DIS_FKT', 'REG_PCODE', 'REGION_NAM', 'BNGRC_R_CO', 'BNGRC_REG_', 'REG_FKT_SH', 'PROV_CODE', 'OLD_PROVIN', 'NOTES', 'OLD_DISTRI', 'SOURCE', 'Shape_Leng', 'Shape_Area', 'geometry']


In [5]:
# Explorer les noms des colonnes pour identifier les champs utiles
print("📋 COLONNES DES SHAPEFILES")
print("=" * 60)

print("\n🗺️ Régions:")
for col in gdf_regions.columns:
    if col != 'geometry':
        sample = gdf_regions[col].iloc[0] if len(gdf_regions) > 0 else 'N/A'
        print(f"   • {col}: {sample}")

print("\n🗺️ Districts:")
for col in gdf_districts.columns:
    if col != 'geometry':
        sample = gdf_districts[col].iloc[0] if len(gdf_districts) > 0 else 'N/A'
        print(f"   • {col}: {sample}")

print("\n🗺️ Communes (échantillon):")
for col in list(gdf_communes.columns)[:8]:
    if col != 'geometry':
        sample = gdf_communes[col].iloc[0] if len(gdf_communes) > 0 else 'N/A'
        print(f"   • {col}: {sample}")

📋 COLONNES DES SHAPEFILES

🗺️ Régions:
   • REG_PCODE: MDG11
   • R_CODE: 11
   • REGION_NAM: Analamanga
   • BNGRC_R_CO: 4.0
   • BNGRC_REG_: ANALAMANGA
   • REG_FKT_SH: ANALAMANGA
   • PROV_CODE: 1
   • OLD_PROVIN: ANTANANARIVO
   • Source: BNGRC (National Disaster Management Office), Region (adm1)  polygons cleaned and merged by OCHA in Dec 2017. BNGRC Region Codes differ from INSTAT Fkt Region codes
   • Notes: Provinces (Faritany) were dissolved in 2007 and Regions became the new Admininistrative Boundary Level 1
   • Shape_Leng: 7.81981093069
   • Shape_Area: 1.48457817466

🗺️ Districts:
   • DIST_PCODE: MDG11101001
   • DISTRICT_N: 1er Arrondissement
   • BNGRC_D_CO: 101.0
   • BNGRC_DIST: 1ER ARRONDISSEMENT
   • DIS_FKT: 1ER ARRONDISSEMENT
   • REG_PCODE: MDG11
   • REGION_NAM: Analamanga
   • BNGRC_R_CO: 4.0
   • BNGRC_REG_: ANALAMANGA
   • REG_FKT_SH: ANALAMANGA
   • C_PROV: 1
   • OLD_PROVIN: ANTANANARIVO
   • NOTES: Old Provinces have been abolished and Regions are now the 

In [6]:
# Afficher les noms des régions
print("📍 RÉGIONS DE MADAGASCAR")
print("=" * 60)

# Identifier la colonne des noms de région
region_col = 'REGION_NAM' if 'REGION_NAM' in gdf_regions.columns else 'ADM1_FR'
regions_list = sorted(gdf_regions[region_col].unique())

print(f"\n{len(regions_list)} régions:")
for i, r in enumerate(regions_list, 1):
    print(f"  {i:2d}. {r}")

📍 RÉGIONS DE MADAGASCAR

22 régions:
   1. Alaotra Mangoro
   2. Amoron I Mania
   3. Analamanga
   4. Analanjirofo
   5. Androy
   6. Anosy
   7. Atsimo Andrefana
   8. Atsimo Atsinanana
   9. Atsinanana
  10. Betsiboka
  11. Boeny
  12. Bongolava
  13. Diana
  14. Haute Matsiatra
  15. Ihorombe
  16. Itasy
  17. Melaky
  18. Menabe
  19. Sava
  20. Sofia
  21. Vakinankaratra
  22. Vatovavy Fitovinany


---
## 4. Chargement du Mapping Zones Côtières ↔ Régions

Nous utilisons le mapping créé dans la Phase 1.2 pour associer les régions administratives aux zones côtières.

In [7]:
# Charger le mapping zones-régions
mapping_file = os.path.join(FINAL_PATH, 'mapping_zones_regions.json')

with open(mapping_file, 'r', encoding='utf-8') as f:
    MAPPING_ZONES_REGIONS = json.load(f)

print("📂 MAPPING ZONES CÔTIÈRES ↔ RÉGIONS")
print("=" * 60)
print(f"\nNombre de zones: {len(MAPPING_ZONES_REGIONS)}")

# Afficher le mapping
for zone, regions in list(MAPPING_ZONES_REGIONS.items())[:5]:
    print(f"\n• {zone}:")
    print(f"  Régions: {regions}")

📂 MAPPING ZONES CÔTIÈRES ↔ RÉGIONS

Nombre de zones: 3

• zones_cotieres:
  Régions: {"CAP D'AMBRE A TOAMASINA": {'cote': 'EST', 'point_debut': "CAP D'AMBRE", 'point_fin': 'TOAMASINA', 'lat_min': -18.5, 'lat_max': -11.5, 'lon_min': 49.0, 'lon_max': 50.5, 'description': "Côte nord-est, de la pointe nord jusqu'à Toamasina"}, "CAP D'AMBRE A MAHANORO": {'cote': 'EST', 'point_debut': "CAP D'AMBRE", 'point_fin': 'MAHANORO', 'lat_min': -20.0, 'lat_max': -11.5, 'lon_min': 48.5, 'lon_max': 50.5, 'description': 'Toute la côte est nord'}, "CAP D'AMBRE A ANTALAHA": {'cote': 'EST', 'point_debut': "CAP D'AMBRE", 'point_fin': 'ANTALAHA', 'lat_min': -15.0, 'lat_max': -11.5, 'lon_min': 49.0, 'lon_max': 50.5, 'description': 'Extrême nord-est'}, 'TOAMASINA AU CAP SAINTE MARIE': {'cote': 'EST-SUD', 'point_debut': 'TOAMASINA', 'point_fin': 'CAP SAINTE MARIE', 'lat_min': -25.6, 'lat_max': -18.0, 'lon_min': 45.0, 'lon_max': 49.5, 'description': 'Côte est-sud, de Toamasina à la pointe sud'}, 'TOAMASINA A TAOL

In [8]:
# Créer le mapping inverse: Région → Zones côtières
MAPPING_REGION_ZONES = {}

for zone, regions in MAPPING_ZONES_REGIONS.items():
    for region in regions:
        if region not in MAPPING_REGION_ZONES:
            MAPPING_REGION_ZONES[region] = []
        MAPPING_REGION_ZONES[region].append(zone)

print("📂 MAPPING INVERSE: RÉGION → ZONES")
print("=" * 60)
print(f"\n{len(MAPPING_REGION_ZONES)} régions mappées")

for region, zones in list(MAPPING_REGION_ZONES.items())[:8]:
    print(f"\n• {region}:")
    for z in zones:
        print(f"  → {z}")

📂 MAPPING INVERSE: RÉGION → ZONES

24 régions mappées

• CAP D'AMBRE A TOAMASINA:
  → zones_cotieres
  → mapping

• CAP D'AMBRE A MAHANORO:
  → zones_cotieres
  → mapping

• CAP D'AMBRE A ANTALAHA:
  → zones_cotieres
  → mapping

• TOAMASINA AU CAP SAINTE MARIE:
  → zones_cotieres
  → mapping

• TOAMASINA A TAOLAGNARO:
  → zones_cotieres
  → mapping

• MAHANORO AU CAP SAINTE MARIE:
  → zones_cotieres
  → mapping

• CAP D'AMBRE A BESALAMPY:
  → zones_cotieres
  → mapping

• BESALAMPY A MOROMBE:
  → zones_cotieres
  → mapping


---
## 5. Calcul des Centroïdes Administratifs

Pour géocoder les incidents sans coordonnées, nous calculons les centroïdes (centres géographiques) des districts et communes.

In [9]:
# Calculer les centroïdes des districts
print("📍 CALCUL DES CENTROÏDES")
print("=" * 60)

# Identifier les colonnes de noms
district_col = [c for c in gdf_districts.columns if 'DIST' in c.upper() and 'NAM' in c.upper()]
district_col = district_col[0] if district_col else 'DIST_NAME'

# Calculer les centroïdes
gdf_districts['centroid'] = gdf_districts.geometry.centroid
gdf_districts['lon_centroid'] = gdf_districts['centroid'].x
gdf_districts['lat_centroid'] = gdf_districts['centroid'].y

# Créer un dictionnaire district → coordonnées
centroides_districts = {}
for _, row in gdf_districts.iterrows():
    nom = row[district_col] if district_col in gdf_districts.columns else str(row.iloc[0])
    centroides_districts[nom.upper().strip()] = {
        'lon': row['lon_centroid'],
        'lat': row['lat_centroid']
    }

print(f"\n✅ {len(centroides_districts)} centroïdes de districts calculés")

# Afficher quelques exemples
print("\nExemples:")
for dist, coords in list(centroides_districts.items())[:5]:
    print(f"  • {dist}: ({coords['lon']:.4f}, {coords['lat']:.4f})")

📍 CALCUL DES CENTROÏDES

✅ 119 centroïdes de districts calculés



Exemples:
  • MDG11101001: (47.5116, -18.9058)
  • MDG11101002: (47.5484, -18.9289)
  • MDG11101003: (47.5289, -18.8964)
  • MDG11101004: (47.5139, -18.9334)
  • MDG11101005: (47.5393, -18.8801)


In [10]:
# Calculer les centroïdes des régions
gdf_regions['centroid'] = gdf_regions.geometry.centroid
gdf_regions['lon_centroid'] = gdf_regions['centroid'].x
gdf_regions['lat_centroid'] = gdf_regions['centroid'].y

# Créer un dictionnaire région → coordonnées
region_col = 'REGION_NAM' if 'REGION_NAM' in gdf_regions.columns else 'ADM1_FR'

centroides_regions = {}
for _, row in gdf_regions.iterrows():
    nom = row[region_col]
    centroides_regions[nom.upper().strip()] = {
        'lon': row['lon_centroid'],
        'lat': row['lat_centroid']
    }

print(f"✅ {len(centroides_regions)} centroïdes de régions calculés")

# Afficher quelques exemples
print("\nExemples:")
for reg, coords in list(centroides_regions.items())[:5]:
    print(f"  • {reg}: ({coords['lon']:.4f}, {coords['lat']:.4f})")

✅ 22 centroïdes de régions calculés

Exemples:
  • ANALAMANGA: (47.4232, -18.4251)
  • VAKINANKARATRA: (46.8474, -19.7348)
  • ITASY: (46.8866, -19.0509)
  • BONGOLAVA: (46.1335, -18.6043)
  • HAUTE MATSIATRA: (46.5994, -21.4436)


---
## 6. Définition des Zones Côtières (Bounding Boxes)

Nous définissons les limites géographiques de chaque zone côtière pour assigner les incidents.

In [11]:
# Définition des zones côtières avec leurs limites géographiques
ZONES_COTIERES_BOUNDS = {
    "CAP D'AMBRE A TOAMASINA": {
        'lat_min': -18.5, 'lat_max': -11.5,
        'lon_min': 49.0, 'lon_max': 51.0,
        'cote': 'EST'
    },
    "CAP D'AMBRE A MAHANORO": {
        'lat_min': -20.0, 'lat_max': -11.5,
        'lon_min': 48.5, 'lon_max': 51.0,
        'cote': 'EST'
    },
    "MAHANORO AU CAP SAINTE MARIE": {
        'lat_min': -25.6, 'lat_max': -19.8,
        'lon_min': 45.0, 'lon_max': 49.5,
        'cote': 'SUD-EST'
    },
    "TOAMASINA AU CAP SAINTE MARIE": {
        'lat_min': -25.6, 'lat_max': -18.0,
        'lon_min': 45.0, 'lon_max': 50.0,
        'cote': 'EST-SUD'
    },
    "CAP D'AMBRE A BESALAMPY": {
        'lat_min': -17.0, 'lat_max': -11.5,
        'lon_min': 45.0, 'lon_max': 49.5,
        'cote': 'NORD-OUEST'
    },
    "BESALAMPY A MOROMBE": {
        'lat_min': -21.8, 'lat_max': -16.5,
        'lon_min': 43.0, 'lon_max': 46.0,
        'cote': 'OUEST'
    },
    "MOROMBE AU CAP SAINTE MARIE": {
        'lat_min': -25.6, 'lat_max': -21.5,
        'lon_min': 43.0, 'lon_max': 46.0,
        'cote': 'SUD-OUEST'
    },
    "CAP D'AMBRE A ANTALAHA": {
        'lat_min': -15.0, 'lat_max': -11.5,
        'lon_min': 48.5, 'lon_max': 50.5,
        'cote': 'NORD-EST'
    },
    "ANTALAHA A TOAMASINA": {
        'lat_min': -18.5, 'lat_max': -14.8,
        'lon_min': 49.0, 'lon_max': 50.5,
        'cote': 'EST'
    },
    "NOSY BE ET ENVIRONS": {
        'lat_min': -14.0, 'lat_max': -12.5,
        'lon_min': 47.5, 'lon_max': 49.0,
        'cote': 'NORD-OUEST'
    }
}

print(f"📍 {len(ZONES_COTIERES_BOUNDS)} zones côtières définies avec leurs limites")

📍 10 zones côtières définies avec leurs limites


---
## 7. Fonctions de Géocodage et d'Assignation

In [12]:
def normaliser_nom(nom):
    """Normalise un nom de lieu pour la recherche."""
    if pd.isna(nom):
        return None
    # Convertir en majuscules, supprimer les accents courants
    nom = str(nom).upper().strip()
    # Remplacements courants
    replacements = {
        'ANTANANARIVO': 'ANTANANARIVO',
        'TANA': 'ANTANANARIVO',
        'MAJUNGA': 'MAHAJANGA',
        'TULEAR': 'TOLIARA',
        'TAMATAVE': 'TOAMASINA',
        'FORT DAUPHIN': 'TAOLAGNARO',
        'FORT-DAUPHIN': 'TAOLAGNARO',
        'DIEGO': 'ANTSIRANANA',
        'DIEGO SUAREZ': 'ANTSIRANANA',
    }
    for old, new in replacements.items():
        if old in nom:
            nom = nom.replace(old, new)
    return nom


def geocoder_par_district(district, centroides_dict):
    """
    Tente de géocoder un incident par son district.
    Retourne (lon, lat) ou (None, None).
    """
    if pd.isna(district):
        return None, None
    
    district_norm = normaliser_nom(district)
    
    # Recherche exacte
    if district_norm in centroides_dict:
        coords = centroides_dict[district_norm]
        return coords['lon'], coords['lat']
    
    # Recherche partielle
    for nom, coords in centroides_dict.items():
        if district_norm in nom or nom in district_norm:
            return coords['lon'], coords['lat']
    
    return None, None


def geocoder_par_region(region, centroides_dict):
    """
    Tente de géocoder un incident par sa région.
    Retourne (lon, lat) ou (None, None).
    """
    if pd.isna(region):
        return None, None
    
    region_norm = normaliser_nom(region)
    
    # Recherche exacte
    if region_norm in centroides_dict:
        coords = centroides_dict[region_norm]
        return coords['lon'], coords['lat']
    
    # Recherche partielle
    for nom, coords in centroides_dict.items():
        if region_norm in nom or nom in region_norm:
            return coords['lon'], coords['lat']
    
    return None, None


print("✅ Fonctions de géocodage définies")

✅ Fonctions de géocodage définies


In [13]:
def assigner_zone_par_coords(lon, lat, zones_bounds):
    """
    Assigne une zone côtière à partir des coordonnées.
    Priorise les zones les plus spécifiques (plus petites).
    """
    if pd.isna(lon) or pd.isna(lat):
        return None
    
    zones_candidates = []
    
    for zone_name, bounds in zones_bounds.items():
        if (bounds['lon_min'] <= lon <= bounds['lon_max'] and
            bounds['lat_min'] <= lat <= bounds['lat_max']):
            # Calculer la taille de la zone (pour prioriser les plus petites)
            taille = (bounds['lon_max'] - bounds['lon_min']) * (bounds['lat_max'] - bounds['lat_min'])
            zones_candidates.append((zone_name, taille))
    
    if zones_candidates:
        # Retourner la zone la plus petite (plus spécifique)
        zones_candidates.sort(key=lambda x: x[1])
        return zones_candidates[0][0]
    
    return None


def assigner_zone_par_region(region, mapping_region_zones):
    """
    Assigne une zone côtière à partir de la région.
    Retourne la première zone correspondante.
    """
    if pd.isna(region):
        return None
    
    region_norm = normaliser_nom(region)
    
    # Recherche exacte
    for reg, zones in mapping_region_zones.items():
        if region_norm == reg.upper() or region_norm in reg.upper() or reg.upper() in region_norm:
            return zones[0] if zones else None
    
    return None


print("✅ Fonctions d'assignation de zone définies")

✅ Fonctions d'assignation de zone définies


---
## 8. Application du Géocodage et Assignation des Zones

In [14]:
# Créer une copie pour le traitement
df_processed = df_vrais_incidents.copy()

# Initialiser les nouvelles colonnes
df_processed['lon_geocode'] = df_processed['Longitude'].copy()
df_processed['lat_geocode'] = df_processed['Latitude'].copy()
df_processed['source_geocode'] = 'original'
df_processed['zone_cotiere'] = None

print("🔄 GÉOCODAGE DES INCIDENTS")
print("=" * 60)

# Compteurs
n_original = 0
n_geocode_district = 0
n_geocode_region = 0
n_non_geocode = 0

for idx, row in df_processed.iterrows():
    # Si coordonnées originales disponibles
    if pd.notna(row['Longitude']) and pd.notna(row['Latitude']):
        n_original += 1
        continue
    
    # Tenter le géocodage par district
    lon, lat = geocoder_par_district(row['District'], centroides_districts)
    if lon is not None:
        df_processed.at[idx, 'lon_geocode'] = lon
        df_processed.at[idx, 'lat_geocode'] = lat
        df_processed.at[idx, 'source_geocode'] = 'district'
        n_geocode_district += 1
        continue
    
    # Tenter le géocodage par région
    lon, lat = geocoder_par_region(row['Region'], centroides_regions)
    if lon is not None:
        df_processed.at[idx, 'lon_geocode'] = lon
        df_processed.at[idx, 'lat_geocode'] = lat
        df_processed.at[idx, 'source_geocode'] = 'region'
        n_geocode_region += 1
        continue
    
    n_non_geocode += 1

print(f"\n✅ Résultats du géocodage:")
print(f"   • Coordonnées originales: {n_original}")
print(f"   • Géocodés par district: {n_geocode_district}")
print(f"   • Géocodés par région: {n_geocode_region}")
print(f"   • Non géocodés: {n_non_geocode}")

total_geocode = n_original + n_geocode_district + n_geocode_region
print(f"\n📊 Taux de géocodage: {100*total_geocode/len(df_processed):.1f}%")

🔄 GÉOCODAGE DES INCIDENTS

✅ Résultats du géocodage:
   • Coordonnées originales: 263
   • Géocodés par district: 0
   • Géocodés par région: 0
   • Non géocodés: 6

📊 Taux de géocodage: 97.8%


In [15]:
# Assigner les zones côtières
print("🔄 ASSIGNATION DES ZONES CÔTIÈRES")
print("=" * 60)

n_zone_coords = 0
n_zone_region = 0
n_zone_none = 0

for idx, row in df_processed.iterrows():
    # Tenter par coordonnées
    zone = assigner_zone_par_coords(row['lon_geocode'], row['lat_geocode'], ZONES_COTIERES_BOUNDS)
    if zone:
        df_processed.at[idx, 'zone_cotiere'] = zone
        n_zone_coords += 1
        continue
    
    # Tenter par région
    zone = assigner_zone_par_region(row['Region'], MAPPING_REGION_ZONES)
    if zone:
        df_processed.at[idx, 'zone_cotiere'] = zone
        n_zone_region += 1
        continue
    
    n_zone_none += 1

print(f"\n✅ Résultats de l'assignation:")
print(f"   • Par coordonnées: {n_zone_coords}")
print(f"   • Par région: {n_zone_region}")
print(f"   • Sans zone: {n_zone_none}")

total_zone = n_zone_coords + n_zone_region
print(f"\n📊 Taux d'assignation: {100*total_zone/len(df_processed):.1f}%")

🔄 ASSIGNATION DES ZONES CÔTIÈRES



✅ Résultats de l'assignation:
   • Par coordonnées: 232
   • Par région: 0
   • Sans zone: 37

📊 Taux d'assignation: 86.2%


---
## 9. Vérification et Analyse des Résultats

In [16]:
# Analyse des zones assignées
print("📊 DISTRIBUTION DES INCIDENTS PAR ZONE CÔTIÈRE")
print("=" * 60)

zones_distribution = df_processed['zone_cotiere'].value_counts()
print(zones_distribution)

print(f"\n📍 Nombre de zones distinctes: {df_processed['zone_cotiere'].nunique()}")
print(f"❌ Incidents sans zone: {df_processed['zone_cotiere'].isna().sum()}")

📊 DISTRIBUTION DES INCIDENTS PAR ZONE CÔTIÈRE
zone_cotiere
ANTALAHA A TOAMASINA            73
CAP D'AMBRE A BESALAMPY         72
NOSY BE ET ENVIRONS             23
BESALAMPY A MOROMBE             19
CAP D'AMBRE A ANTALAHA          17
MAHANORO AU CAP SAINTE MARIE    14
MOROMBE AU CAP SAINTE MARIE      7
CAP D'AMBRE A MAHANORO           6
CAP D'AMBRE A TOAMASINA          1
Name: count, dtype: int64

📍 Nombre de zones distinctes: 9
❌ Incidents sans zone: 37


In [17]:
# Examiner les incidents sans zone
df_sans_zone = df_processed[df_processed['zone_cotiere'].isna()]

if len(df_sans_zone) > 0:
    print("❌ INCIDENTS SANS ZONE ASSIGNÉE")
    print("=" * 60)
    
    print(f"\nNombre: {len(df_sans_zone)}")
    print("\nInformations disponibles:")
    display(df_sans_zone[['Date debut', 'District', 'Commune', 'Region', 'Types', 'description']].head(10))

❌ INCIDENTS SANS ZONE ASSIGNÉE

Nombre: 37

Informations disponibles:


,Date debut,District,Commune,Region,Types,description
65,07/03/2017,OCEAN INDIEN,NaN,NaN,NaN,"Le vraquier IRIS II (IMO 9286906, dwt 75798, c..."
89,31/03/2017,OCEAN INDIEN,NaN,NaN,NaN,Un marin philippin âgé de 48 ans a été découve...
189,09/07/2017,OCEAN INDIEN,NaN,NaN,NaN,"Un navire grec ""AEGEAN ANGEL"" a demandé de l'..."
205,25/07/2017,OCEAN INDIEN,NaN,NaN,NaN,"A la position 29°23.00 S / 045°49.00 E, un mem..."
219,07/08/2017,OCEAN INDIEN,NaN,NaN,NaN,Le palangrier de thon HSIANG FUH 6 (Pavillon T...
256,13/09/2017,OCEAN INDIEN,NaN,NaN,NaN,Le vraquier MV TINA (IMO 9215749) a rencontré...
267,24/09/2017,OCEAN INDIEN,NaN,NaN,NaN,"Le navire cargo M/V AT 40, pavillon Belize, a ..."
286,13/10/2017,MADAGASCAR,NaN,NaN,NaN,"Le navire pétrolier et chimiquier WIGMORE, pav..."
320,16/11/2017,CANAL DE MOZAMBIQUE,NaN,NaN,NaN,"Un yacht sud-africain, le KINDA MAGIC a connu ..."
347,13/12/2017,CANAL DU MOZAMBIQUE,NaN,NaN,NaN,"À la position 16°04S 042°E, le BW EAGLE (IMO 9..."


In [18]:
# Analyse des types d'incidents par zone
print("📊 TYPES D'INCIDENTS PAR ZONE CÔTIÈRE")
print("=" * 60)

df_avec_zone = df_processed[df_processed['zone_cotiere'].notna()]

# Crosstab zones x types
crosstab = pd.crosstab(df_avec_zone['zone_cotiere'], df_avec_zone['Types'])
display(crosstab)

📊 TYPES D'INCIDENTS PAR ZONE CÔTIÈRE


Types,,ACC,ACCIDENTS,ACCIDENT_CADAVRE,ACCIDENT_DERIVE,ACCIDENT_DISPARITION,ACCIDENT_NAUFRAGE,ACCIDENT_NOYADE,ACC_,ACC_CADAVRE,...,ACC_DISPARITION,ACC_ECHOUAGE,ACC_INCENDIE,ACC_NAUFRAGE,ACC_NOYADE,ACC_PANNE,ASSISTANCE,POLMAR,Recif corallien:Marée basse,SAR
zone_cotiere,,,,,,,,,,,,,,,,,,,,,
ANTALAHA A TOAMASINA,0,0,1,1,0,1,1,1,0,1,...,0,1,2,5,4,1,1,0,0,3
BESALAMPY A MOROMBE,0,0,0,0,0,0,0,0,0,0,...,1,0,0,1,1,0,0,0,0,0
CAP D'AMBRE A ANTALAHA,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,2,0,1,0,1
CAP D'AMBRE A BESALAMPY,1,0,1,1,1,0,0,1,1,1,...,1,1,1,1,0,0,0,0,1,3
CAP D'AMBRE A MAHANORO,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1
MAHANORO AU CAP SAINTE MARIE,1,0,0,0,0,0,0,1,0,0,...,0,1,0,2,0,0,0,0,0,0
MOROMBE AU CAP SAINTE MARIE,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
NOSY BE ET ENVIRONS,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,2


---
## 10. Préparation et Sauvegarde du Dataset Final

In [19]:
# Préparer le dataset final
print("📝 PRÉPARATION DU DATASET FINAL")
print("=" * 60)

# Parser la date
df_processed['date_incident'] = pd.to_datetime(df_processed['Date debut'], format='%d/%m/%Y', errors='coerce')
df_processed['annee'] = df_processed['date_incident'].dt.year
df_processed['mois'] = df_processed['date_incident'].dt.month
df_processed['jour'] = df_processed['date_incident'].dt.day
df_processed['jour_semaine'] = df_processed['date_incident'].dt.dayofweek

# Ajouter un flag pour la saison cyclonique (novembre à avril)
df_processed['saison_cyclonique'] = df_processed['mois'].isin([11, 12, 1, 2, 3, 4]).astype(int)

# Sélectionner et renommer les colonnes
colonnes_finales = [
    'date_incident', 'annee', 'mois', 'jour', 'jour_semaine', 'saison_cyclonique',
    'zone_cotiere', 'Region', 'District', 'Commune',
    'lon_geocode', 'lat_geocode', 'source_geocode',
    'Types', 'description',
    'Personne concerne', 'Mort'
]

df_final = df_processed[[c for c in colonnes_finales if c in df_processed.columns]].copy()

# Renommer pour plus de clarté
df_final.rename(columns={
    'Types': 'type_incident',
    'Region': 'region',
    'District': 'district',
    'Commune': 'commune',
    'Personne concerne': 'personnes_concernees',
    'Mort': 'deces'
}, inplace=True)

print("\nColonnes du dataset final:")
for col in df_final.columns:
    print(f"  • {col}")

print(f"\n📊 Dimensions: {df_final.shape}")

📝 PRÉPARATION DU DATASET FINAL

Colonnes du dataset final:
  • date_incident
  • annee
  • mois
  • jour
  • jour_semaine
  • saison_cyclonique
  • zone_cotiere
  • region
  • district
  • commune
  • lon_geocode
  • lat_geocode
  • source_geocode
  • type_incident
  • description
  • personnes_concernees
  • deces

📊 Dimensions: (269, 17)


In [20]:
# Afficher un aperçu
print("📋 APERÇU DU DATASET FINAL")
print("=" * 60)

display(df_final.head(15))

📋 APERÇU DU DATASET FINAL


,date_incident,annee,mois,jour,jour_semaine,saison_cyclonique,zone_cotiere,region,district,commune,lon_geocode,lat_geocode,source_geocode,type_incident,description,personnes_concernees,deces
35,2017-02-05,2017,2,5,6,1,CAP D'AMBRE A BESALAMPY,NaN,MADAGASCAR,NaN,46.325890,-15.766938,original,NaN,"A Mahajanga, le cadavre d’un homme retrouvé au...",NaN,NaN
55,2017-02-25,2017,2,25,5,1,CAP D'AMBRE A BESALAMPY,NaN,Katsepy Mahajanga/ Madagascar,NaN,46.289816,-15.737907,original,NaN,Le bac Makuba a coulé 25 Fév,NaN,NaN
65,2017-03-07,2017,3,7,1,1,None,NaN,OCEAN INDIEN,NaN,54.740555,-21.401068,original,NaN,"Le vraquier IRIS II (IMO 9286906, dwt 75798, c...",NaN,NaN
75,2017-03-17,2017,3,17,4,1,BESALAMPY A MOROMBE,NaN,madagascar,NaN,43.872715,-18.042964,original,NaN,Un boutre qui a quitté Morondava pour aller à...,NaN,NaN
89,2017-03-31,2017,3,31,4,1,None,NaN,OCEAN INDIEN,NaN,68.761516,-20.827302,original,NaN,Un marin philippin âgé de 48 ans a été découve...,NaN,NaN
106,2017-04-17,2017,4,17,0,1,ANTALAHA A TOAMASINA,NaN,MADAGASCAR,NaN,49.420532,-18.150623,original,NaN,Un garçon de 14 ans vient de se noyer en mer q...,NaN,NaN
109,2017-04-20,2017,4,20,3,1,CAP D'AMBRE A ANTALAHA,NaN,MADAGASCAR,NaN,50.063335,-13.360009,original,NaN,MADAGASCAR,NaN,NaN
128,2017-05-09,2017,5,9,1,0,ANTALAHA A TOAMASINA,NaN,MADAGASCAR,NaN,49.869580,-15.966218,original,NaN,"Le bateau à moteur FIDELYS, avec à son bord de...",NaN,NaN
189,2017-07-09,2017,7,9,6,0,None,NaN,OCEAN INDIEN,NaN,59.665862,13.417318,original,NaN,"Un navire grec ""AEGEAN ANGEL"" a demandé de l'...",NaN,NaN
191,2017-07-11,2017,7,11,1,0,ANTALAHA A TOAMASINA,NaN,MADAGASCAR,NaN,49.502451,-17.514099,original,NaN,"Le MS ATLANTIS II, avec 9 équipages à bord et ...",NaN,NaN


In [21]:
# Sauvegarder le dataset
output_file = os.path.join(FINAL_PATH, 'incidents_geocodes.csv')

# Formater la date pour l'export
df_export = df_final.copy()
df_export['date_incident'] = df_export['date_incident'].dt.strftime('%Y-%m-%d')

df_export.to_csv(output_file, index=False, encoding='utf-8')

print("💾 SAUVEGARDE EFFECTUÉE")
print("=" * 60)
print(f"\n✅ Fichier sauvegardé: {output_file}")
print(f"📊 Nombre d'incidents: {len(df_export)}")
print(f"📍 Incidents avec zone: {df_export['zone_cotiere'].notna().sum()}")
print(f"📅 Période: {df_export['annee'].min()} à {df_export['annee'].max()}")

💾 SAUVEGARDE EFFECTUÉE

✅ Fichier sauvegardé: /home/henintsoa/CFIM/final/data/incidents_geocodes.csv
📊 Nombre d'incidents: 269
📍 Incidents avec zone: 232
📅 Période: 2017 à 2022


---
## 11. Résumé et Statistiques Finales

In [22]:
# Statistiques finales
print("\n" + "="*70)
print("                    RÉSUMÉ FINAL - GÉOCODAGE")
print("="*70)

print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║                       RÉSULTATS DU GÉOCODAGE                         ║
╠══════════════════════════════════════════════════════════════════════╣
║  📊 Incidents traités: {len(df_vrais_incidents)}                                            ║
║  ✅ Avec coordonnées originales: {n_original}                                    ║
║  📍 Géocodés par district: {n_geocode_district}                                        ║
║  📍 Géocodés par région: {n_geocode_region}                                         ║
║  ❌ Non géocodés: {n_non_geocode}                                                ║
║                                                                      ║
║  📈 Taux de géocodage total: {100*total_geocode/len(df_processed):.1f}%                              ║
╠══════════════════════════════════════════════════════════════════════╣
║                    ASSIGNATION DES ZONES                             ║
╠══════════════════════════════════════════════════════════════════════╣
║  📍 Zones assignées par coordonnées: {n_zone_coords}                              ║
║  📍 Zones assignées par région: {n_zone_region}                                   ║
║  ❌ Sans zone: {n_zone_none}                                                    ║
║                                                                      ║
║  📈 Taux d'assignation total: {100*total_zone/len(df_processed):.1f}%                             ║
╠══════════════════════════════════════════════════════════════════════╣
║                     FICHIER DE SORTIE                                ║
╠══════════════════════════════════════════════════════════════════════╣
║  💾 {output_file.split('/')[-1]:53} ║
╚══════════════════════════════════════════════════════════════════════╝
""")

print("\n✅ Notebook 2.2 terminé avec succès!")


                    RÉSUMÉ FINAL - GÉOCODAGE

╔══════════════════════════════════════════════════════════════════════╗
║                       RÉSULTATS DU GÉOCODAGE                         ║
╠══════════════════════════════════════════════════════════════════════╣
║  📊 Incidents traités: 269                                            ║
║  ✅ Avec coordonnées originales: 263                                    ║
║  📍 Géocodés par district: 0                                        ║
║  📍 Géocodés par région: 0                                         ║
║  ❌ Non géocodés: 6                                                ║
║                                                                      ║
║  📈 Taux de géocodage total: 97.8%                              ║
╠══════════════════════════════════════════════════════════════════════╣
║                    ASSIGNATION DES ZONES                             ║
╠══════════════════════════════════════════════════════════════════════╣
║  📍 Zones assig

---
## Prochaines Étapes

### ✅ Ce qui a été réalisé
1. **Géocodage** des incidents sans coordonnées à partir des districts et régions
2. **Assignation** de chaque incident à une zone côtière
3. **Enrichissement** avec des variables temporelles (année, mois, saison cyclonique)

### ➡️ Prochaine étape
**Notebook 2.3**: Création du dataset d'entraînement
- Fusion incidents + météo par date et zone
- Gestion des jours sans incident (classe négative)
- Préparation des features pour le modèle